# Imports

In [ ]:
# System functionality
import os
import glob 

# Data Analysis
import xarray as xr
import xesmf as xe
import numpy as np
import pandas as pd

# Plotting
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib import colors as mcolors
from matplotlib import ticker as mticker

# Cartopy
from cartopy import crs as ccrs
from cartopy import feature as cfeature
from cartopy import util as cutil
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LongitudeLocator, LatitudeLocator

# Auxiliary Functions
from auxiliary_functions.time_utils import datetime64_to_yyyymmdd, string_to_yyyymm, convert_time_to_ns, convert_ns_to_datetime, extract_years_months
from auxiliary_functions import xarray_utils

# Logging
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
)
logger = logging.getLogger(__name__)

In [ ]:
start_date = "2020-12-29T00:00:00.000000000"
end_date = "2021-02-27T00:00:00.000000000"
logger.info(f"Start date: {start_date}")
logger.info(f"End date: {end_date}")

logger.info("Starting data download script")
def surface_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
    # Subset to a specific region and keep only one variable
    subset_data = ds.sel(
        # level=pressure_levels,
        time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
    )

    target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
    regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

    return regridder(subset_data) # type: ignore

def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
    # Subset to a specific region and keep only one variable
    subset_data = ds.sel(
        level=pressure_levels,
        time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
    ).assign_coords({'level': pressure_levels.astype(np.int32)})

    target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
    regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

    return regridder(subset_data) # type: ignore

yyyymm_strings = pd.date_range(
    pd.to_datetime(start_date).to_period("M").to_timestamp(),
    pd.to_datetime(end_date).to_period("M").to_timestamp(),
    freq="MS"
).strftime("%Y%m")

logger.info(f"Dates: {np.datetime64(start_date).astype('datetime64[h]')} : {np.datetime64(end_date).astype('datetime64[h]')}")

graphcast_data_directory = f"/glade/u/home/sressel/spencer-scratch/graphcast_input_data/{string_to_yyyymm(start_date)}_{string_to_yyyymm(end_date)}"
if not os.path.exists(graphcast_data_directory):
    logger.info("Creating output directory...")
    os.makedirs(graphcast_data_directory, exist_ok=True)
logger.info(f"Graphcast data directory: {graphcast_data_directory}")

logger.info("Load all data into a single dataset")
all_data = xr.open_mfdataset(f"{graphcast_data_directory}/*.nc")
logger.info(f"    Saving data...")
all_data.to_netcdf(f"{graphcast_data_directory}/era5_data.nc")
logger.info("Finished")

In [ ]:
years, months = extract_years_months(start_date, end_date)

# Surface level variables
logger.info("Surface level variables")
surface_base = "/gdex/data/d633000/e5.oper.an.sfc"

surface_variables = {
    "2m_temperature": "2t",
    "mean_sea_level_pressure": "msl",
    "10m_u_component_of_wind": "10u",
    "10m_v_component_of_wind": "10v"
}

surface_variables_old_names = {
    "2m_temperature": "VAR_2T",
    "mean_sea_level_pressure": "MSL",
    "10m_u_component_of_wind": "VAR_10U",
    "10m_v_component_of_wind": "VAR_10V"
}

for variable in surface_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{surface_base}/{ym}/e5.oper.an.sfc.*_{surface_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))


# Load RMM data

In [ ]:
def load_txt_as_xarray(path):
    # Read the whitespace-delimited file
    df = pd.read_csv(
        path,
        delim_whitespace=True,
        header=None,
        names=["year", "month", "day", "hour", "var1", "var2", "var3"],
        na_values=[-99.0],
    )

    # Build datetime64 index
    df["time"] = pd.to_datetime(df[["year", "month", "day", "hour"]])

    # Set time as index
    df = df.set_index("time")

    # Drop the original date columns
    df = df.drop(columns=["year", "month", "day", "hour"])

    # Convert to xarray Dataset
    ds = xr.Dataset.from_dataframe(df)

    return ds

# Example usage:
RMM_indices = load_txt_as_xarray("RMM.txt")
RMM1 = RMM_indices["var1"]
RMM2 = RMM_indices["var2"]
amplitude = RMM_indices["var3"]


In [ ]:
def convert_time_to_ns(ds):
    # Original datetime64 coordinate
    datetime = ds["time"]

    # Compute nanoseconds since first timestep
    t0 = datetime.values[0]
    time_ns = (datetime.values - t0).astype("timedelta64[ns]").astype("timedelta64[ns]")

    new_datetime = datetime.expand_dims('batch').assign_coords(
        datetime=("time", datetime.values),   # secondary coordinate
        time=("time", time_ns)                # replace primary coordinate
    )

    # Assign new coordinates
    ds = ds.expand_dims('batch').assign_coords(
        datetime=new_datetime,
        time=("time", time_ns)
    )

    return ds

In [ ]:
start_date = '1992-08-14T00:00:00.000000000'
end_date = '1992-11-12T00:00:00.000000000'
yyyymm_strings = pd.date_range(
    pd.to_datetime(start_date).to_period("M").to_timestamp(),
    pd.to_datetime(end_date).to_period("M").to_timestamp(),
    freq="MS"
).strftime("%Y%m")


def pressure_level_preprocess(ds: xr.Dataset) -> xr.Dataset:
    # Subset to a specific region and keep only one variable
    subset_data = ds.sel(
        level=pressure_levels,
        time=ds.time.where(ds['time'].dt.hour.isin([0, 6, 12, 18]), drop=True)
    ).assign_coords({'level': pressure_levels.astype(np.int32)})

    target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
    regridder = xe.Regridder(subset_data, target_grid, "bilinear", reuse_weights=False) 

    return regridder(subset_data) # type: ignore

pressure_level_base = "/gdex/data/d633000/e5.oper.an.pl"

pressure_levels = xr.DataArray(
    data = [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000],
    dims=['level'],
    coords={'level': [50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000]}
)

pressure_level_variables = {
    "geopotential": "z",
    # "temperature": "t",
    # "u_component_of_wind": "u",
    # "v_component_of_wind": "v",
    # "specific_humidity": "q",
    # "vertical_velocity": "w",
}

pressure_level_variables_old_names = {
    "geopotential": "Z",
    "temperature": "T",
    "u_component_of_wind": "U",
    "v_component_of_wind": "V",
    "specific_humidity": "Q",
    "vertical_velocity": "W",
}

for variable in pressure_level_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{pressure_level_base}/{ym}/e5.oper.an.pl.*_{pressure_level_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))

    if not files_list:
        logger.info(f"No files found for variable {variable} in month {ym}")
    else:
        logger.info(f"    Loading files...")
        pressure_level_data = xr.open_mfdataset(sorted(files_list)[:3], preprocess=pressure_level_preprocess).load()


In [ ]:
# yyyymm_strings
pressure_level_data.coords['time']
# sorted(files_list)

# xr.open_dataset("/gdex/data/d633000/e5.oper.an.pl/199208/e5.oper.an.pl.128_130_t.ll025sc.1992080100_1992080123.nc")

In [ ]:
target_duration

In [ ]:
# plt.scatter(
#     RMM1.isel(time=slice(3371-100, 3371+100)),
#     RMM2.isel(time=slice(3371-100, 3371+100)),
# )
# plt.xlim(-4, 4)
# plt.ylim(-4, 4)
# plt.gca().set_aspect('equal')

# Configure plot
[fig, ax] = plt.subplots(figsize=(9,9))
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["axes.linewidth"] = 3
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
plt.xlabel("RMM1")
plt.ylabel("RMM2")
ax.set_facecolor("white")

# Plot index points
colormap = sns.color_palette("viridis", as_cmap=True)
start_time = '2009-10-01T00:00:00.000000000'
end_time = '2010-04-30T00:00:00.000000000'
start_index = list(amplitude.time.values).index(amplitude.sel(time=start_time).time)
end_index = list(amplitude.time.values).index(amplitude.sel(time=end_time).time)
ax.plot(RMM1[start_index], RMM2[start_index], color="black", marker=".", ls="-", ms=30)
for i in range(start_index + 1, end_index + 1):
    ax.plot(
        RMM1[i],
        RMM2[i],
        color=colormap((i - start_index) / (end_index - start_index)),
        marker="o",
        ms=10,
    )

# Add phase regions overlay
circle1 = plt.Circle((0, 0), 1.0, color="k", fill=False, lw=3, zorder=10)
ax.hlines(y=0, xmin=-4, xmax=-1, color="k", lw=3, ls="-")
ax.hlines(y=0, xmin=1, xmax=4, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=-4, ymax=-1, color="k", lw=3, ls="-")
ax.vlines(x=0, ymin=1, ymax=4, color="k", lw=3, ls="-")
ax.plot([np.sqrt(2) / 2, 4], [np.sqrt(2) / 2, 4], color="k", lw=3, ls="-")


x_vals = {
    1:-1,
    2:-1,
    3:1,
    4:1,
    5:1,
    6:1,
    7:-1,
    8:-1
}

y_val1 = {
    1:0,
    2:-4,
    3:-4,
    4:0,
    5:0,
    6:np.linspace(0,4,len(RMM1)),
    7:np.linspace(0,4,len(RMM1)),
    8:0
}

y_val2 = {
    1:-1*np.linspace(0, 4, len(RMM1)),
    2:-1*np.linspace(0, 4, len(RMM1)),
    3:-1*np.linspace(0, 4, len(RMM1)),
    4:-1*np.linspace(0, 4, len(RMM1)),
    5:1*np.linspace(0, 4, len(RMM1)),
    6:4,
    7:4,
    8:1*np.linspace(0, 4, len(RMM1))
}

# Fill one of the phases in with blue
# val=2
# ax.fill_between(
#     x_vals[val]*np.linspace(0, 4, len(RMM1)), 
#     y_val1[val], 
#     y_val2[val]
# )

# x = np.linspace(-1,1,100)
# ax.fill_between(x, -np.sqrt(1-x**2), np.sqrt(1-x**2), color='white')

# Add lines to differentiate the phases
ax.plot([np.sqrt(2) / 2, 4], [-np.sqrt(2) / 2, -4], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [4, np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.plot([-4, -np.sqrt(2) / 2], [-4, -np.sqrt(2) / 2], color="k", lw=3, ls="-")
ax.add_patch(circle1)

# Add phase labels
ax.text(-3.5, -0.26, f'Phase 1',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, -3.75, f'Phase 2', horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, -3.75, f'Phase 3',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, -0.26, f'Phase 4',   horizontalalignment='center',
     verticalalignment='center')
ax.text(3.5, 0.25, f'Phase 5',    horizontalalignment='center',
     verticalalignment='center')
ax.text(0.5, 3.75, f'Phase 6',    horizontalalignment='center',
     verticalalignment='center')
ax.text(-0.51, 3.75, f'Phase 7',  horizontalalignment='center',
     verticalalignment='center')
ax.text(-3.5, 0.25, f'Phase 8',   horizontalalignment='center',
     verticalalignment='center')

# Add MJO-location labels
# ax.text(0, 3.5, f'Western Pacific',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(-3.3, 0, f'Western Hemisphere \n & Africa',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(3.3, 0, f'Maritime Continent',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

# ax.text(0, -3.5, f'Indian Ocean',  horizontalalignment='center',
#      verticalalignment='center', bbox=props, fontsize=14)

ax.set_aspect("equal")
plt.tight_layout()

plt.show()

# Process ERA5 data

## Download ERA5 data

In [ ]:
# start_time = "2021-05-01"
# end_time = "2022-12-31"

start_time = "1974-00-01"
end_time = "1979-12-31"

variable = 'zonal_wind'

variable_name = {
    'zonal_wind':'u_component_of_wind',
    'meridional_wind':'v_component_of_wind'
    }

from datetime import datetime
from dateutil.relativedelta import relativedelta

def iter_year_months(start, end):
    """
    Yield (year, month) pairs for all months between two arbitrary dates.
    Accepts datetime objects or ISO-format strings.
    """

    # Parse strings if needed
    if isinstance(start, str):
        start = datetime.fromisoformat(start)
    if isinstance(end, str):
        end = datetime.fromisoformat(end)

    # Normalize to first-of-month
    cursor = start.replace(day=1)
    end_month = end.replace(day=1)

    while cursor <= end_month:
        yield cursor.year, cursor.month
        cursor += relativedelta(months=1)

import cdsapi

c = cdsapi.Client()

for year, month in iter_year_months(start_time, end_time):
    print(year, month)
    output_file = f"/glade/derecho/scratch/sressel/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{year}_{month}_raw.nc"
    c.retrieve(
        "reanalysis-era5-pressure-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                variable_name[variable],
                # "v_component_of_wind",
            ],
            "pressure_level": [
                "100", "125", "150", "175", "200", "225", "250", "300", "350", "400",
                "450", "500", "550", "600", "650", "700", "750", "775", "800", "825",
                "850", "875", "900", "925", "950", "975", "1000",
            ],
            "year": year,
            "month": month,
            "day": [
                "01","02","03","04","05","06","07","08","09","10",
                "11","12","13","14","15","16","17","18","19","20",
                "21","22","23","24","25","26","27","28","29","30","31",
            ],
            "time": ["00:00"],   # daily at 00 UTC
            "area": [30, -180, -30, 180],  # North, West, South, East
            "grid": [2.5, 2.5],  # 2.5° x 2.5°
            "format": "netcdf",
        },
        output_file,
    )

# data_processed = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{start_time.year}_{end_time.year}_raw.nc").rename(
#     {
#         'longitude': 'lon',
#         'latitude': 'lat',
#         'valid_time': 'time',
#         'pressure_level': 'plev',
#     }
# ).transpose("time", "plev", "lat", "lon").isel(plev=slice(None, None, -1)).sortby('plev').sortby('lat').coord_funcs.lon_to_360('lon')
# data_processed.to_netcdf("/glade/u/home/sressel/spencer-scratch/ECMWF/ERA5/daily_data/daily_25_degree_{variable}_{start_time.year}_{end_time.year}.nc")


# Load Graphcast sample data

In [ ]:
gc_data = xr.open_dataset("/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/data/era5_sample.nc")
gc_data.coords['datetime']

In [ ]:
from typing import Any, Mapping, Sequence, Tuple, Union

TimedeltaLike = Any  # Something convertible to pd.Timedelta.
TimedeltaStr = str  # A string convertible to pd.Timedelta.

TargetLeadTimes = Union[
    TimedeltaLike,
    Sequence[TimedeltaLike],
    slice  # with TimedeltaLike as its start and stop.
]
def _process_target_lead_times_and_get_duration(
    target_lead_times: TargetLeadTimes) -> TimedeltaLike:
  """Returns the minimum duration for the target lead times."""
  if isinstance(target_lead_times, slice):
    # A slice of lead times. xarray already accepts timedelta-like values for
    # the begin/end/step of the slice.
    if target_lead_times.start is None:
      # If the start isn't specified, we assume it starts at the next timestep
      # after lead time 0 (lead time 0 is the final input timestep):
      target_lead_times = slice(
          pd.Timedelta(1, "ns"), target_lead_times.stop, target_lead_times.step
      )
    target_duration = pd.Timedelta(target_lead_times.stop)
  else:
    if not isinstance(target_lead_times, (list, tuple, set)):
      # A single lead time, which we wrap as a length-1 array to ensure there
      # still remains a time dimension (here of length 1) for consistency.
      target_lead_times = [target_lead_times]

    # A list of multiple (not necessarily contiguous) lead times:
    target_lead_times = [pd.Timedelta(x) for x in target_lead_times]
    target_lead_times.sort()
    target_duration = target_lead_times[-1]
  return target_lead_times, target_duration

target_lead_times = slice("6h", f"{40*6}h")
(target_lead_times, target_duration
   ) = _process_target_lead_times_and_get_duration(target_lead_times)
target_duration
time = gc_data.coords["time"]
new_gc_data = gc_data.assign_coords(time=time + target_duration - time[-1])

In [ ]:
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
params_file_path = "/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/params/params_GraphCast_small.npz"

def load_params(params_file_path=None):
    if params_file_path is None:
        params_file_path = os.environ.get('GRAPHCAST_PARAMS_PATH', 'params/params_GraphCast_small.npz')

    with open(params_file_path, "rb") as f:
        ckpt = checkpoint.load(f, graphcast.CheckPoint)
        return ckpt.params, ckpt.model_config, ckpt.task_config, ckpt.description
    
params, model_config, task_config, model_description = load_params(params_file_path="/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/params/params_GraphCast_small.npz")

In [ ]:
np.load("/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/params/params_GraphCast_small.npz")['task_config:input_duration']
# task_config.input_duration

In [ ]:
# Surface level variables
logger.info("Surface level variables")
surface_base = "/gdex/data/d633000/e5.oper.an.sfc"

surface_variables = {
    "2m_temperature": "2t",
    # "mean_sea_level_pressure": "msl",
    # "10m_u_component_of_wind": "10u",
    # "10m_v_component_of_wind": "10v"
}

surface_variables_old_names = {
    "2m_temperature": "VAR_2T",
    # "mean_sea_level_pressure": "MSL",
    # "10m_u_component_of_wind": "VAR_10U",
    # "10m_v_component_of_wind": "VAR_10V"
}

for variable in surface_variables.keys():
    files_list = []

    logger.info(f"  {variable}")
    for ym in yyyymm_strings:
        pattern = f"{surface_base}/{ym}/e5.oper.an.sfc.*_{surface_variables[variable]}.*.nc"
        files_list.extend(glob.glob(pattern))

    if not files_list:
        print(f"No files found for variable {variable} in month {ym}")
    else:
        logger.info(f"    Loading files...")
        surface_data = convert_time_to_ns(xr.open_mfdataset(sorted(files_list), preprocess=surface_level_preprocess).load())
        surface_data = surface_data.rename({surface_variables_old_names[variable]: variable})
        datetimes = surface_data.datetime


In [ ]:
datetimes

In [ ]:
# import cdsapi
# # Precipitation
# logger.info("Precipitation")
# target = f"{graphcast_data_directory}/total_precipitation_6hr.nc"
# dataset = "reanalysis-era5-single-levels"
# request = {
#     "product_type": ["reanalysis"],
#     "variable": [
#         "total_precipitation",
#     ],
#     "year": years,
#     "month": months,
#     "day": [
#         "01", "02", "03",
#         "04", "05", "06",
#         "07", "08", "09",
#         "10", "11", "12",
#         "13", "14", "15",
#         "16", "17", "18",
#         "19", "20", "21",
#         "22", "23", "24",
#         "25", "26", "27",
#         "28", "29", "30",
#         "31"
#     ],
#     "time": [
#         "00:00", "01:00", "02:00",
#         "03:00", "04:00", "05:00",
#         "06:00", "07:00", "08:00",
#         "09:00", "10:00", "11:00",
#         "12:00", "13:00", "14:00",
#         "15:00", "16:00", "17:00",
#         "18:00", "19:00", "20:00",
#         "21:00", "22:00", "23:00"
#     ],
#     "data_format": "netcdf",
#     "download_format": "unarchived"
# }

# client = cdsapi.Client()
# logger.info("    Downloading data...")
# client.retrieve(dataset, request, target)

# logger.info("    Regridding data...")
# raw_precipitation_files = sorted(glob.glob(f"{graphcast_data_directory}/total_precipitation_6hr.nc"))
# precipitation_data = xr.open_mfdataset(raw_precipitation_files)['tp'].rename({'valid_time': 'time'}).load()

# six_hour_accumulated_precipitation = precipitation_data.resample(time='6h').sum()

# target_grid = xr.Dataset(
#         {
#             "lat": (["lat"], np.arange(-90, 91, 1.0)),
#             "lon": (["lon"], np.arange(0, 360, 1.0)),
#         }
#     )
# regridder = xe.Regridder(six_hour_accumulated_precipitation, target_grid, "bilinear", reuse_weights=False)
# regridded_precipitation = regridder(six_hour_accumulated_precipitation)
# regridded_precipitation = convert_time_to_ns(regridded_precipitation.sel(time=datetimes.sel(batch=0).values))

# regridded_precipitation.name = 'total_precipitation_6hr'
# resave_data = True
# if resave_data:
#     logger.info(f"    Output directory: {graphcast_data_directory}")
#     logger.info(f"    Saving data...")
#     regridded_precipitation.to_netcdf(f"{graphcast_data_directory}/total_precipitation_6hr.nc", mode='w')
#     regridded_precipitation.close()
#     del regridded_precipitation
#     gc.collect()

# logger.info("Finished")

# TOA Solar Radiation
logger.info("TOA Solar Radiation")
target = f"{graphcast_data_directory}/toa_incident_solar_radiation.nc"
dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "toa_incident_solar_radiation"
    ],
    "year": years,
    "month": months,
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00","12:00","18:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived"
}

import gc
client = cdsapi.Client()
logger.info("    Downloading data...")
client.retrieve(dataset, request, target)

logger.info("    Regridding data...")
toa_incident_solar_radiation_data = xr.open_mfdataset(target)['tisr'].rename({'valid_time':'time'}).load()

target_grid = xr.Dataset(
        {
            "lat": (["lat"], np.arange(-90, 91, 1.0)),
            "lon": (["lon"], np.arange(0, 360, 1.0)),
        }
    )
regridder = xe.Regridder(toa_incident_solar_radiation_data, target_grid, "bilinear", reuse_weights=False)
regridded_toa_incident_solar_radiation = regridder(toa_incident_solar_radiation_data)
regridded_toa_incident_solar_radiation = convert_time_to_ns(regridded_toa_incident_solar_radiation.sel(time=datetimes.sel(batch=0).values))
regridded_toa_incident_solar_radiation.name = 'toa_incident_solar_radiation'

resave_data = True
if resave_data:
    logger.info(f"    Output directory: {graphcast_data_directory}")
    logger.info(f"    Saving data...")
    regridded_toa_incident_solar_radiation.to_netcdf(f"{graphcast_data_directory}/toa_incident_solar_radiation.nc", mode='w')
    regridded_toa_incident_solar_radiation.close()
    del regridded_toa_incident_solar_radiation
    gc.collect()


In [ ]:
# era5_data = xr.open_dataset(f"{graphcast_data_directory}/era5_data.nc")
# precipitation_data = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102/total_precipitation_6hr.nc")['tp'].rename({'valid_time': 'time'}).load()
# six_hour_accumulated_precipitation = precipitation_data.resample(time='6h').sum()

# target_grid = xr.Dataset(
#         {
#             "lat": (["lat"], np.arange(-90, 91, 1.0)),
#             "lon": (["lon"], np.arange(0, 360, 1.0)),
#         }
#     )
# regridder = xe.Regridder(six_hour_accumulated_precipitation, target_grid, "bilinear", reuse_weights=False)
# regridded_precipitation = regridder(six_hour_accumulated_precipitation)
regridded_precipitation = convert_time_to_ns(regridded_precipitation.sel(time=datetimes.sel(batch=0).values))

In [ ]:
regridded_precipitation

In [ ]:
datetimes.sel(batch=0)

In [ ]:
data

In [ ]:
all_data = xr.open_mfdataset(f"{graphcast_data_directory}/*.nc", combine='by_coords')

In [ ]:
import cdsapi
start_date="2020-12-29T00:00:00.000000000"
end_date="2021-02-27T00:00:00.000000000"

years, months = extract_years_months(start_date, end_date)
# TOA Solar Radiation
logger.info("TOA Solar Radiation")
target = f"{graphcast_data_directory}/toa_incident_solar_radiation_test.nc"
dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "toa_incident_solar_radiation"
    ],
    "year": years,
    "month": months,
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "06:00","12:00","18:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived"
}

client = cdsapi.Client()
logger.info("    Downloading data...")
client.retrieve(dataset, request, target)

In [ ]:
# data = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102/era5_data.nc")
plt.contourf(
    data['specific_humidity'].lon,
    data['specific_humidity'].time,
    data['total_precipitation_6hr'].sel(batch=0, lat=slice(-15,15)).mean(dim='lat').stats.standardize(dim='time'),
    levels=21,
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0)
)
plt.colorbar()

In [ ]:
data


In [ ]:
# input_duration = np.load("/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/params/params_GraphCast_small.npz")['task_config:input_duration']
input_duration = '12h'
# Slice out targets:
targets = new_gc_data.sel({"time": target_lead_times})

input_duration = pd.Timedelta(input_duration)
# Both endpoints are inclusive with label-based slicing, so we offset by a
# small epsilon to make one of the endpoints non-inclusive:
zero = pd.Timedelta(0)
epsilon = pd.Timedelta(1, "ns")
inputs = new_gc_data.sel({"time": slice(-input_duration + epsilon, zero)})

In [ ]:
_SEC_PER_HOUR = 3600
_HOUR_PER_DAY = 24
SEC_PER_DAY = _SEC_PER_HOUR * _HOUR_PER_DAY
_AVG_DAY_PER_YEAR = 365.24219
AVG_SEC_PER_YEAR = SEC_PER_DAY * _AVG_DAY_PER_YEAR

DAY_PROGRESS = "day_progress"
YEAR_PROGRESS = "year_progress"
_DERIVED_VARS = {
    DAY_PROGRESS,
    f"{DAY_PROGRESS}_sin",
    f"{DAY_PROGRESS}_cos",
    YEAR_PROGRESS,
    f"{YEAR_PROGRESS}_sin",
    f"{YEAR_PROGRESS}_cos",
}
TISR = "toa_incident_solar_radiation"
def extract_inputs_targets_forcings(
    dataset: xr.Dataset,
    *,
    input_variables: Tuple[str, ...],
    target_variables: Tuple[str, ...],
    forcing_variables: Tuple[str, ...],
    pressure_levels: Tuple[int, ...],
    input_duration: TimedeltaLike,
    target_lead_times: TargetLeadTimes,
    ) -> Tuple[xr.Dataset, xr.Dataset, xr.Dataset]:
  """Extracts inputs, targets and forcings according to requirements."""
  dataset = dataset.sel(level=list(pressure_levels))

  # "Forcings" include derived variables that do not exist in the original ERA5
  # or HRES datasets, as well as other variables (e.g. tisr) that need to be
  # computed manually for the target lead times. Compute the requested ones.
  if set(forcing_variables) & _DERIVED_VARS:
    add_derived_vars(dataset)
  if set(forcing_variables) & {TISR}:
    add_tisr_var(dataset)

  # `datetime` is needed by add_derived_vars but breaks autoregressive rollouts.
  dataset = dataset.drop_vars("datetime")

  inputs, targets = extract_input_target_times(
      dataset,
      input_duration=input_duration,
      target_lead_times=target_lead_times)

  if set(forcing_variables) & set(target_variables):
    raise ValueError(
        f"Forcing variables {forcing_variables} should not "
        f"overlap with target variables {target_variables}."
    )

  inputs = inputs[list(input_variables)]
  # The forcing uses the same time coordinates as the target.
  forcings = targets[list(forcing_variables)]
  targets = targets[list(target_variables)]

  return inputs, targets, forcings

In [ ]:
from xarray import load_dataset as xld
from numpy import datetime64
dataset_file_path = "/glade/u/home/sressel/thesis-work/python/graphcast/gc-initial-condition-optimization/data/era5_sample.nc"

init_date = '2022-01-01T06:00:00.000000000'
init_str = init_date[:13]  # Extract the first 13 characters (YYYY-MM-DDTHH)
init_date = datetime64(init_date)
name_path = dataset_file_path.split('/')[-1][5:15]
inter = xld(dataset_file_path).compute()

def drop_time_dim_if_present(ds, var_name):
    if 'time' in ds[var_name].dims:
        return ds[var_name].isel(time=0, drop=True)
    return ds[var_name]

inter = inter.assign(
    geopotential_at_surface=drop_time_dim_if_present(inter, 'geopotential_at_surface'),
    land_sea_mask=drop_time_dim_if_present(inter, 'land_sea_mask')
)
for j,i in enumerate(inter.datetime.values[0]):
    if i==init_date:
        start_index=j-1
        break

example_batch = inter.isel(time=slice(start_index,None))

In [ ]:
from logging import info
date_train=str(example_batch.datetime.values[0][1])[:13]
info(date_train)
num_time_steps = len(example_batch.time)
new_time = np.arange(0, num_time_steps * 6 * 3600 * 10**9, 6 * 3600 * 10**9, dtype='timedelta64[ns]')
new_time

In [ ]:
inter.swap_dims({'time':'datetime'}).sel(datetime=slice(np.datetime64(init_date)-np.timedelta64(6, "h"),None))

# Analyze optimized data

In [ ]:
# data_directory = "/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102"
# files_list = glob.glob(f"{data_directory}/*.nc")
# for file in files_list:
#     data_name = file.split("/")[-1].split("202012_202102_")[-1]
#     print(data_name)
#     os.system(f"mv {file} {data_name}")

In [ ]:
import xarray as xr
input_data = xr.open_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/202012_202102/era5_data.nc")

In [ ]:
optimized_data_array = xr.zeros_like(input_data.isel(time=0, drop=True))
optimized_data = np.load("/glade/u/home/sressel/spencer-scratch/graphcast_output/perfect_model_params/2009-12-26T00/2009-12-26T00_96_40.npz")

In [ ]:
# Assign variables with inferred dims
for key in [
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    '2m_temperature',
    # 'day_progress_cos',
    # 'day_progress_sin',
    'geopotential',
    'geopotential_at_surface',
    'land_sea_mask',
    'mean_sea_level_pressure',
    'specific_humidity',
    'temperature',
    'toa_incident_solar_radiation',
    'total_precipitation_6hr',
    'u_component_of_wind',
    'v_component_of_wind',
    'vertical_velocity'
]:

    arr = optimized_data[key]

    if arr.ndim == 2:
        optimized_data_array[key] = (("lat", "lon"), arr)

    elif arr.ndim == 3:
        optimized_data_array[key] = (("time", "lat", "lon"), arr)

    elif arr.ndim == 4:
        optimized_data_array[key] = (("batch", "time", "lat", "lon"), arr)

    elif arr.ndim == 5:
        optimized_data_array[key] = (("batch", "time", "level", "lat", "lon"), arr)

    else:
        raise ValueError(f"Don't know how to assign dims for {key} with shape {arr.shape}")

In [ ]:
def datetime_to_ns(initial_time, final_time):
    return (final_time.astype("timedelta64[ns]") - initial_time.astype("timedelta64[ns]")).astype("timedelta64[ns]")

In [ ]:
start_date = '2009-12-01T00:00:00.000000000'
end_date = '2010-01-31T00:00:00.000000000'

optimized_day = datetime_to_ns(
    np.datetime64(start_date), np.datetime64('2009-12-26T00:00:00.000000000')
)
level_to_plot = 850
variable_to_plot = 'specific_humidity'

# [fig, ax] = plt.subplots(3, 1, figsize=(12,16))

fig, ax = plt.subplots(nrows=3,ncols=1,
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        figsize=(11,8.5))

i  =0
c_opt = ax[0].contourf(
    optimized_data_array[variable_to_plot].lon,
    optimized_data_array[variable_to_plot].lat,
    optimized_data_array[variable_to_plot].sel(level=level_to_plot).isel(batch=0, time=i),
    levels = 21
)
fig.colorbar(c_opt, ax=ax[0])
ax[0].add_feature(cartopy.feature.COASTLINE, linewidth=0.5)

c_init = ax[1].contourf(
    input_data.lon,
    input_data.lat,
    input_data[variable_to_plot].sel(time=optimized_day, level=level_to_plot).isel(batch=0),
    levels = c_opt.levels
)
fig.colorbar(c_init, ax=ax[1])
ax[1].add_feature(cartopy.feature.COASTLINE, linewidth=0.5)

c_diff = ax[2].contourf(
    input_data.lon,
    input_data.lat,
    (optimized_data_array[variable_to_plot].sel(level=level_to_plot).isel(time=i, batch=0) - input_data[variable_to_plot].sel(time=optimized_day, level=level_to_plot).isel(batch=0)),
    # levels = np.linspace(-0.01, 0.01, 21),
    levels=21,
    cmap = 'coolwarm'
)
fig.colorbar(c_diff, ax=ax[2])
ax[2].add_feature(cartopy.feature.COASTLINE, linewidth=0.5)

# for axis in ax:
#     rect = patches.Rectangle(
#         (70, -10),        # lower-left corner
#         30, 20,   # rectangle size
#         linewidth=1.5,
#         edgecolor='red',
#         facecolor='none',   # or a color like 'lightgray'
#         alpha=0.5
#     )

#     axis.add_patch(rect)

for axis in ax:
    axis.set_xlim(60, 110)
    axis.set_ylim(-15, 15)

plt.show()

In [ ]:
# pred_data = xr.open_zarr("/glade/u/home/sressel/spencer-scratch/graphcast_output/perfect_model_forecasts/2009-12-26T00/2009-12-26T00_40.zarr")
pred_data = xr.open_zarr("/glade/u/home/sressel/spencer-scratch/graphcast_output/perfect_model_forecasts/2021-01-01T00/2021-01-01T00_200.zarr")

start_date = '2020-01-01T00:00:00.000000000'
# end_date = '2021-11-12T00:00:00.000000000'

# optimized_day = datetime_to_ns(
#     np.datetime64(start_date), np.datetime64('2009-12-26T00:00:00.000000000')
# )
level_to_plot = 850
variable_to_plot = 'specific_humidity'

# [fig, ax] = plt.subplots(3, 1, figsize=(12,16))

fig, ax = plt.subplots(nrows=3,ncols=1,
                        subplot_kw={'projection': ccrs.PlateCarree()},
                        figsize=(11,8.5))

i=-1
c_input = ax[0].contourf(
    input_data[variable_to_plot].lon,
    input_data[variable_to_plot].lat,
    input_data[variable_to_plot].sel(level=level_to_plot).isel(batch=0, time=i),
    levels = 21
)
fig.colorbar(c_input, ax=ax[0])
ax[0].add_feature(cfeature.COASTLINE, linewidth=0.5)

c_pred = ax[1].contourf(
    pred_data.lon,
    pred_data.lat,
    pred_data[variable_to_plot].sel(level=level_to_plot).isel(batch=0, time=i),
    levels = c_input.levels
)
fig.colorbar(c_pred, ax=ax[1])
ax[1].add_feature(cfeature.COASTLINE, linewidth=0.5)

c_diff = ax[2].contourf(
    input_data.lon,
    input_data.lat,
    (pred_data[variable_to_plot] - input_data[variable_to_plot]).sel(level=level_to_plot).isel(batch=0, time=i),
    # levels = np.linspace(-0.01, 0.01, 21),
    levels=21,
    cmap = 'coolwarm'
)
fig.colorbar(c_diff, ax=ax[2])
ax[2].add_feature(cfeature.COASTLINE, linewidth=0.5)

for axis in ax:
    axis.set_xlim(60, 110)
    axis.set_ylim(-15, 15)

# for axis in ax:
#     rect = patches.Rectangle(
#         (70, -10),        # lower-left corner
#         30, 20,   # rectangle size
#         linewidth=1.5,
#         edgecolor='red',
#         facecolor='none',   # or a color like 'lightgray'
#         alpha=0.5
#     )

#     axis.add_patch(rect)
plt.show()

In [ ]:


# from datetime import datetime
# datetime.fromisoformat()

In [ ]:
predicted_precipitation = pred_data['total_precipitation_6hr'].load().isel(batch=0)
init_time = "2021-01-01T00:00:00"
deltas = predicted_precipitation['time'].astype("timedelta64[ns]")
datetimes = np.datetime64(init_time) + deltas
predicted_precipitation = predicted_precipitation.assign_coords(datetime=("time", datetimes.values))

In [ ]:
input_data

In [ ]:
# input_data_subset = input_data.where(input_data.datetime.isin(predicted_precipitation.datetime)).dropna(dim='time').isel(batch=0)

fig = plt.figure(figsize=(16,9))
gs = GridSpec(2, 3, figure=fig, height_ratios=[30,1])
gs.update(top=1, bottom=0, left=0, right=1, wspace=0.2)

axes = [
    fig.add_subplot(gs[0,0]),
    fig.add_subplot(gs[0,1]),
    fig.add_subplot(gs[0,2]),
]
cbar_axes = [
    fig.add_subplot(gs[1, :2]),
    fig.add_subplot(gs[1, -1])
]

im_era5 = axes[0].contourf(
    input_data_subset['total_precipitation_6hr'].lon,
    input_data_subset['total_precipitation_6hr'].time,
    (1000/6)*(input_data_subset['total_precipitation_6hr'].sel(lat=slice(-10,10)).mean(dim='lat')),
    levels=np.arange(0, 2.25, 0.25)
)
im_pred = axes[1].contourf(
    predicted_precipitation.lon,
    predicted_precipitation.time,
    (1000/6)*(predicted_precipitation.sel(lat=slice(-10,10)).mean(dim='lat')),
    levels=im_era5.levels
)
fig.colorbar(im_era5, cax=cbar_axes[0], orientation='horizontal')
im_diff = axes[2].contourf(
    input_data_subset['total_precipitation_6hr'].lon,
    input_data_subset['total_precipitation_6hr'].time,
    (1000/6)*(
        input_data_subset['total_precipitation_6hr'].sel(lat=slice(-10,10)).mean(dim='lat').values
        - predicted_precipitation.sel(lat=slice(-10,10)).mean(dim='lat').values
    ),
    cmap='coolwarm',
    levels=np.arange(-2, 2.5, 0.5),
    norm=mcolors.CenteredNorm(vcenter=0)
)
fig.colorbar(im_diff, cax=cbar_axes[1], orientation='horizontal')

# for ax in axes:
#     ax.set_yticks(predicted_precipitation.time[::4])#labels=predicted_precipitation.datetime[::4*10])
plt.show()

In [ ]:
np.min((1000/6)*input_data_subset['total_precipitation_6hr'].sel(lat=slice(-10,10)).mean(dim='lat').values
        - (1000/6)*predicted_precipitation.sel(lat=slice(-10,10)).mean(dim='lat').values)

# Analyze predicted data

In [ ]:
era5_data = xr.load_dataset("/glade/u/home/sressel/spencer-scratch/graphcast_input_data/199208_199211/era5_data.nc").load()
era5_data = convert_ns_to_datetime(era5_data, '1992-08-01').swap_dims({'time':'datetime'})
pred_data = xr.open_zarr("/glade/u/home/sressel/spencer-scratch/graphcast_output/perfect_model_forecasts/1992-08-11T00/1992-08-11T00_120.zarr").load()
# pred_data = xr.open_zarr("/glade/u/home/sressel/spencer-scratch/graphcast_output/perfect_model_forecasts/1992-08-14T00/1992-08-14T00_364.zarr")
pred_data = convert_ns_to_datetime(pred_data, "1992-08-10T18").swap_dims({'time':'datetime'})

In [ ]:
plt.contourf(pred_data['total_precipitation_6hr'].isel(batch=0).sel(lat=slice(-5,5)).mean(dim='lat'))

In [ ]:
variable_to_plot = 'total_precipitation_6hr'
time_to_plot = "1992-08-11T06"

plt.rcParams.update({'font.size':16})
fig = plt.figure(figsize=(16,9))
gs = GridSpec(1, 3, figure=fig, width_ratios=[100, 100, 10])
gs.update(top=1, bottom=0, left=0, right=1, wspace=0.2)

central_longitude = -180
proj = ccrs.PlateCarree(central_longitude=central_longitude)
data_crs = ccrs.PlateCarree()
coastline_width = 1


ax = [
    fig.add_subplot(gs[0], projection=proj),
    fig.add_subplot(gs[1], projection=proj),
]
cbar_ax = fig.add_subplot(gs[2])

# cdata_era5 = xarray_utils.add_cyclic_point(
#     4000*era5_data['total_precipitation_6hr'],
#     dim='lon'
# ).isel(batch=0).sel(lat=slice(-30, 30), lon=slice(60, 180), datetime="1992-08-14T06"),

# cdata_pred = xarray_utils.add_cyclic_point(
#     4000*pred_data['total_precipitation_6hr'],
#     dim='lon'
# ).isel(batch=0).sel(lat=slice(-30, 30), lon=slice(60, 180), datetime="1992-08-14T06"),

im = ax[0].contourf(
    era5_data.lon,
    era5_data.lat,
    (4000*era5_data[variable_to_plot]).isel(batch=0).sel(datetime=time_to_plot),
    levels=np.arange(0, 40, 5), 
    # levels=np.linspace(-21, 21, 21),
    transform=data_crs,
    cmap='coolwarm', 
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='max'
)

ax[1].contourf(
    pred_data.lon,
    pred_data.lat,
    (4000*pred_data[variable_to_plot]).isel(batch=0).sel(datetime=time_to_plot),
    # levels=np.arange(0, 27.5, 2.5), 
    levels=im.levels, 
    transform=data_crs,
    cmap='coolwarm', 
    norm=mcolors.CenteredNorm(vcenter=0),
    extend='max'
)

# Add colorbar
cbar = fig.colorbar(im, cax=cbar_ax)

for axis in ax:
    axis.add_feature(cfeature.COASTLINE, lw=coastline_width)
    axis.set_xlim(central_longitude+60, central_longitude+180)
    axis.set_ylim(-30, 30)
    gl = axis.gridlines(
        crs=proj,
        draw_labels=True,
        linewidth=1,
        color="gray",
        alpha=0.75,
        linestyle="-",
        zorder=15
    )
    gl.right_labels = False
    gl.top_labels = False
    gl.xlocator = mticker.FixedLocator(central_longitude+np.arange(60,210,15))
    # gl.xlocator = LongitudeLocator(30)
    gl.xformatter = LongitudeFormatter()
    gl.xlabel_style = {'fontsize':20}
    gl.ylocator = mticker.FixedLocator(np.arange(-40,40,10))
    gl.yformatter = LatitudeFormatter()
    gl.ylabel_style = {'fontsize':20}

plt.show()

In [ ]:
meridional_mean_graphcast_precip = pred_data['total_precipitation_6hr'].sel(lat=slice(-15,15)).mean(dim='lat')
meridional_mean_era5_precip = era5_data['total_precipitation_6hr'].sel(lat=slice(-15,15)).mean(dim='lat')

In [ ]:
print(4000*meridional_mean_graphcast_precip.isel(batch=0).min().values, 4000*meridional_mean_graphcast_precip.isel(batch=0).max().values)

In [ ]:
plt.rcParams.update({'font.size':16})
fig = plt.figure(figsize=(16,9))
gs = GridSpec(1, 3, figure=fig, width_ratios=[100, 100, 10])
gs.update(top=1, bottom=0, left=0, right=1, wspace=0.2)

ax = [
    fig.add_subplot(gs[0]),
    fig.add_subplot(gs[1]),
    fig.add_subplot(gs[2]),
]

plotting_dates = slice(
    "1992-08-15",
    "1992-11-13"
)

ax[0].set_title('ERA5')
ax[0].contourf(
    meridional_mean_era5_precip.lon,
    meridional_mean_era5_precip.datetime.sel(datetime=plotting_dates),
    4000*meridional_mean_era5_precip.isel(batch=0).sel(datetime=plotting_dates),
    levels=np.arange(0, 27.5, 2.5),
    extend='both',
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0)
)

ax[1].set_title('Graphcast Prediction')
im = ax[1].contourf(
    meridional_mean_graphcast_precip.lon,
    meridional_mean_graphcast_precip.datetime.sel(datetime=plotting_dates),
    4000*meridional_mean_graphcast_precip.isel(batch=0).sel(datetime=plotting_dates),
    levels=np.arange(0, 27.5, 2.5),
    extend='both',
    cmap='coolwarm',
    norm=mcolors.CenteredNorm(vcenter=0)
)

ax[1].set_yticklabels('')

fig.colorbar(im, cax=ax[-1])

plt.show()

In [ ]:
meridional_mean_era5_precip.isel(batch=0)

In [ ]:
optimized_data.files